In [15]:
# 모드버스접속
from pymodbus.client import ModbusTcpClient
import time as tt
import threading
client = ModbusTcpClient('210.119.14.58', port=502)
client.connect()

True

In [16]:
# 초기화
client.write_coils(0, [0]*16 )

In [17]:
# 켜야되는 출력 2부터 7개 모두 On
client.write_coils(2, [1]*7)

In [18]:
# 클래스로 2공정 묶기

class Vscan:
    def __init__(self, client, bit1, bit2, coil):
        self.client = client
        self.scan_run = True
        self.bit1 = bit1
        self.bit2 = bit2
        self.coil = coil
        self.mem = 0

    def run(self):
        while self.scan_run:
            bits = self.client.read_discrete_inputs(0, count=8).bits
            if bits[self.bit1]:
                if self.mem == 0:
                    self.mem = tt.monotonic()
                elif tt.monotonic() - self.mem >= 1:
                    self.client.write_coil(self.coil,1) # Pusher On
                    self.mem = 0
            else:
                self.mem = 0  
            if bits[self.bit2]: 
                self.client.write_coil(self.coil,0) # Pusher Off
                self.mem = 0
            tt.sleep(0.1)

    def start(self):
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.thread.start()
        print("쓰레드 시작")
    def stop(self):
        self.scan_run = False

In [19]:
vscan1 = Vscan(client, 0, 3, 0)
vscan2 = Vscan(client, 1, 5, 1)
vscan1.start()
vscan2.start()

쓰레드 시작
쓰레드 시작
